In [1]:
from openpyxl import load_workbook
from openpyxl.cell.cell import MergedCell
from openpyxl.utils import range_boundaries
from copy import copy
from collections import Counter
from datetime import datetime, date, time, timedelta
import re


INPUT_FILE = r"../data/raw/mta2024to2026/raw_unmerged/Störliste STW-Mittelteilanlage 2026.xlsx"
OUTPUT_FILE = r"../data/raw/mta2024to2026/raw_unmerged/Störliste STW-Mittelteilanlage 2026_filled.xlsx"

SHEET_NAME = None  # None = aktives Tabellenblatt
BLOCK_SIZE = 3
PRINT_DEBUG_BLOCKS = False


def normalize_header(value):
    if value is None:
        return ""

    text = str(value).lower()
    text = text.replace("\n", "")
    text = text.replace("\r", "")
    text = text.replace("\xa0", "")
    text = re.sub(r"[\s\-\.\(\)]", "", text)
    return text


def is_empty(value):
    return value is None or str(value).strip() == ""


def parse_date_value(value):
    if value is None:
        return None

    if isinstance(value, datetime):
        return value.date()

    if isinstance(value, date):
        return value

    text = str(value).strip()

    for fmt in ("%d.%m.%Y", "%Y-%m-%d", "%d/%m/%Y"):
        try:
            return datetime.strptime(text, fmt).date()
        except ValueError:
            pass

    return None


def parse_time_value(value):
    if value is None:
        return None

    if isinstance(value, datetime):
        return value.time()

    if isinstance(value, time):
        return value

    if isinstance(value, (int, float)):
        seconds = int(round((value % 1) * 24 * 60 * 60))
        hours = seconds // 3600
        minutes = (seconds % 3600) // 60
        seconds = seconds % 60
        return time(hours % 24, minutes, seconds)

    text = str(value).strip()

    for fmt in ("%H:%M:%S", "%H:%M"):
        try:
            return datetime.strptime(text, fmt).time()
        except ValueError:
            pass

    return None


def find_column(ws, header_row, search_text):
    search = normalize_header(search_text)

    for col in range(1, ws.max_column + 1):
        header = normalize_header(ws.cell(header_row, col).value)

        if search in header:
            return col

    return None


def find_column_any(ws, header_row, search_texts):
    for search_text in search_texts:
        col = find_column(ws, header_row, search_text)
        if col:
            return col
    return None


def find_header_row(ws):
    for row in range(1, 31):
        if (
            find_column(ws, row, "Datum")
            and find_column(ws, row, "Schicht")
            and find_column(ws, row, "Menge Gesamt")
        ):
            return row

    return None


def create_raw_snapshot(ws):
    snapshot = {}

    for row in range(1, ws.max_row + 1):
        for col in range(1, ws.max_column + 1):
            snapshot[(row, col)] = ws.cell(row, col).value

    return snapshot


def create_expanded_snapshot(ws, raw_snapshot):
    """
    Logischer Snapshot:
    Verbundene Zellen werden intern so behandelt,
    als stünde der Wert in jeder Zelle des verbundenen Bereichs.
    Die Datei wird dabei noch nicht verändert.
    """
    snapshot = dict(raw_snapshot)

    for merged_range in ws.merged_cells.ranges:
        min_col, min_row, max_col, max_row = range_boundaries(str(merged_range))
        top_left_value = raw_snapshot.get((min_row, min_col))

        for row in range(min_row, max_row + 1):
            for col in range(min_col, max_col + 1):
                snapshot[(row, col)] = top_left_value

    return snapshot


def snapshot_value(snapshot, row, col):
    return snapshot.get((row, col))


def original_date_exists_in_previous_rows(raw_snapshot, row, datum_col, header_row, lookback=3):
    start_row = max(header_row + 1, row - lookback)

    for check_row in range(start_row, row):
        value = parse_date_value(snapshot_value(raw_snapshot, check_row, datum_col))

        if value is not None:
            return True

    return False


def get_first_time_in_block(expanded_snapshot, block_rows, zeit_von_col):
    for row in block_rows:
        value = parse_time_value(snapshot_value(expanded_snapshot, row, zeit_von_col))
        if value is not None:
            return value

    return None


def row_has_relevant_content(snapshot, row, min_col, max_col):
    for col in range(min_col, max_col + 1):
        if not is_empty(snapshot_value(snapshot, row, col)):
            return True
    return False


def detect_blocks_from_merged_ranges(ws, header_row, min_col, max_col):
    """
    Hauptlogik:
    Blöcke werden aus den originalen vertikal verbundenen 3-Zeilen-Bereichen erkannt.
    Dadurch kann der Block nicht um eine Zeile verrutschen.
    """
    block_counter = Counter()

    for merged_range in ws.merged_cells.ranges:
        merged_min_col, merged_min_row, merged_max_col, merged_max_row = range_boundaries(str(merged_range))

        if merged_min_row <= header_row:
            continue

        height = merged_max_row - merged_min_row + 1

        if height != BLOCK_SIZE:
            continue

        intersects_target_area = not (
            merged_max_col < min_col
            or merged_min_col > max_col
        )

        if not intersects_target_area:
            continue

        block_rows = (merged_min_row, merged_min_row + 1, merged_min_row + 2)
        block_counter[block_rows] += 1

    return block_counter


def detect_fallback_blocks_from_repeated_times(
    expanded_snapshot,
    header_row,
    max_row,
    min_col,
    max_col,
    zeit_von_col,
    zeit_bis_col,
    used_rows
):
    """
    Fallback:
    Falls Bereiche nicht verbunden sind, werden 3 direkt aufeinanderfolgende
    Zeilen mit gleichem Zeitfenster als Block erkannt.

    Diese Logik wird nur für Zeilen genutzt, die nicht bereits über Merged Cells erkannt wurden.
    """
    fallback_blocks = []

    row = header_row + 1

    while row + 2 <= max_row:
        candidate_rows = [row, row + 1, row + 2]

        if any(r in used_rows for r in candidate_rows):
            row += 1
            continue

        if not any(row_has_relevant_content(expanded_snapshot, r, min_col, max_col) for r in candidate_rows):
            row += 1
            continue

        time_keys = []

        for r in candidate_rows:
            zeit_von = parse_time_value(snapshot_value(expanded_snapshot, r, zeit_von_col))
            zeit_bis = parse_time_value(snapshot_value(expanded_snapshot, r, zeit_bis_col))

            if zeit_von is not None or zeit_bis is not None:
                time_keys.append((zeit_von, zeit_bis))

        unique_time_keys = []
        for key in time_keys:
            if key not in unique_time_keys:
                unique_time_keys.append(key)

        if len(unique_time_keys) == 1:
            fallback_blocks.append(tuple(candidate_rows))
            used_rows.update(candidate_rows)
            row += 3
        else:
            row += 1

    return fallback_blocks


def choose_non_overlapping_blocks(block_scores):
    """
    Entfernt überlappende Block-Kandidaten.
    Bei Konflikten gewinnt der Block mit höherem Score.
    """
    candidates = sorted(
        block_scores.items(),
        key=lambda item: (item[0][0], -item[1])
    )

    selected = []

    for block_rows, score in candidates:
        block_set = set(block_rows)

        overlap_index = None
        for i, (selected_rows, selected_score) in enumerate(selected):
            if block_set & set(selected_rows):
                overlap_index = i
                break

        if overlap_index is None:
            selected.append((block_rows, score))
        else:
            existing_rows, existing_score = selected[overlap_index]
            if score > existing_score:
                selected[overlap_index] = (block_rows, score)

    return [list(rows) for rows, score in sorted(selected, key=lambda item: item[0][0])]


def unmerge_and_expand_values(ws, min_row, max_row, min_col, max_col):
    """
    Löst verbundene Zellen im Zielbereich auf.
    Der Wert der linken oberen Zelle wird in alle Zellen des verbundenen Bereichs geschrieben.
    """
    merged_ranges = list(ws.merged_cells.ranges)

    for merged_range in merged_ranges:
        range_string = str(merged_range)
        merged_min_col, merged_min_row, merged_max_col, merged_max_row = range_boundaries(range_string)

        intersects_target_area = not (
            merged_max_row < min_row
            or merged_min_row > max_row
            or merged_max_col < min_col
            or merged_min_col > max_col
        )

        if not intersects_target_area:
            continue

        top_left_cell = ws.cell(merged_min_row, merged_min_col)
        value = top_left_cell.value

        font = copy(top_left_cell.font)
        fill = copy(top_left_cell.fill)
        border = copy(top_left_cell.border)
        alignment = copy(top_left_cell.alignment)
        number_format = top_left_cell.number_format
        protection = copy(top_left_cell.protection)

        ws.unmerge_cells(range_string)

        for row in range(merged_min_row, merged_max_row + 1):
            for col in range(merged_min_col, merged_max_col + 1):
                if row < min_row or row > max_row:
                    continue

                if col < min_col or col > max_col:
                    continue

                cell = ws.cell(row, col)
                cell.value = value
                cell.font = copy(font)
                cell.fill = copy(fill)
                cell.border = copy(border)
                cell.alignment = copy(alignment)
                cell.number_format = number_format
                cell.protection = copy(protection)


def get_original_dates_in_block(raw_snapshot, block_rows, datum_col):
    dates = []

    for row in block_rows:
        value = parse_date_value(snapshot_value(raw_snapshot, row, datum_col))

        if value is not None and value not in dates:
            dates.append(value)

    return dates


def determine_block_dates(raw_snapshot, expanded_snapshot, blocks, header_row, datum_col, zeit_von_col):
    """
    Ermittelt je Block das korrekte Datum.

    Regeln:
    - Vorhandene Original-Datumswerte gewinnen immer.
    - Wenn im Block kein Datum steht, wird das letzte Datum übernommen.
    - +1 Tag nur, wenn:
        Zeit von kleiner als vorherige Zeit von ist
        UND in den drei Zeilen darüber in der Ursprungsdatei kein Datum stand.
    """
    block_dates = {}
    conflicts = []

    current_date = None
    last_time_von = None

    for block_rows in blocks:
        original_dates = get_original_dates_in_block(raw_snapshot, block_rows, datum_col)
        block_start = block_rows[0]
        block_time = get_first_time_in_block(expanded_snapshot, block_rows, zeit_von_col)

        if len(original_dates) > 1:
            conflicts.append(
                {
                    "rows": f"{block_rows[0]}-{block_rows[-1]}",
                    "column": "Datum",
                    "values": original_dates,
                }
            )
            continue

        if len(original_dates) == 1:
            current_date = original_dates[0]
            block_dates[tuple(block_rows)] = current_date

            if block_time is not None:
                last_time_von = block_time

            continue

        if current_date is None:
            conflicts.append(
                {
                    "rows": f"{block_rows[0]}-{block_rows[-1]}",
                    "column": "Datum",
                    "values": "Kein vorheriges Datum vorhanden",
                }
            )
            continue

        if block_time is not None:
            time_goes_back = last_time_von is not None and block_time < last_time_von

            no_original_date_above = not original_date_exists_in_previous_rows(
                raw_snapshot=raw_snapshot,
                row=block_start,
                datum_col=datum_col,
                header_row=header_row,
                lookback=3
            )

            if time_goes_back and no_original_date_above:
                current_date = current_date + timedelta(days=1)

            last_time_von = block_time

        block_dates[tuple(block_rows)] = current_date

    return block_dates, conflicts


def get_unique_values_in_block(raw_snapshot, expanded_snapshot, block_rows, col, datum_col, block_date):
    values = []

    for row in block_rows:
        if col == datum_col:
            original_date = parse_date_value(snapshot_value(raw_snapshot, row, datum_col))
            value = original_date if original_date is not None else block_date
        else:
            value = snapshot_value(expanded_snapshot, row, col)

        if not is_empty(value) and value not in values:
            values.append(value)

    return values


def fill_blocks(ws):
    header_row = find_header_row(ws)

    if header_row is None:
        raise ValueError("Kopfzeile mit 'Datum', 'Schicht' und 'Menge Gesamt' wurde nicht gefunden.")

    datum_col = find_column(ws, header_row, "Datum")
    schicht_col = find_column(ws, header_row, "Schicht")
    zeit_von_col = find_column_any(ws, header_row, ["Zeit von", "Zeitvon", "Zeit vo"])
    zeit_bis_col = find_column_any(ws, header_row, ["Zeit bis", "Zeitbis", "Zeit bi"])
    end_col = find_column(ws, header_row, "Menge Gesamt")

    if zeit_von_col is None:
        raise ValueError("Spalte 'Zeit von' wurde nicht gefunden.")

    if zeit_bis_col is None:
        raise ValueError("Spalte 'Zeit bis' wurde nicht gefunden.")

    raw_snapshot = create_raw_snapshot(ws)
    expanded_snapshot = create_expanded_snapshot(ws, raw_snapshot)

    merged_block_scores = detect_blocks_from_merged_ranges(
        ws=ws,
        header_row=header_row,
        min_col=datum_col,
        max_col=end_col
    )

    used_rows = set()
    for block_rows in merged_block_scores:
        used_rows.update(block_rows)

    fallback_blocks = detect_fallback_blocks_from_repeated_times(
        expanded_snapshot=expanded_snapshot,
        header_row=header_row,
        max_row=ws.max_row,
        min_col=datum_col,
        max_col=end_col,
        zeit_von_col=zeit_von_col,
        zeit_bis_col=zeit_bis_col,
        used_rows=used_rows
    )

    block_scores = Counter()

    for block_rows, score in merged_block_scores.items():
        block_scores[block_rows] += score * 10

    for block_rows in fallback_blocks:
        block_scores[block_rows] += 1

    blocks = choose_non_overlapping_blocks(block_scores)

    block_dates, date_conflicts = determine_block_dates(
        raw_snapshot=raw_snapshot,
        expanded_snapshot=expanded_snapshot,
        blocks=blocks,
        header_row=header_row,
        datum_col=datum_col,
        zeit_von_col=zeit_von_col
    )

    unmerge_and_expand_values(
        ws=ws,
        min_row=header_row + 1,
        max_row=ws.max_row,
        min_col=datum_col,
        max_col=end_col
    )

    conflicts = list(date_conflicts)
    processed_blocks = []

    for block_rows in blocks:
        block_key = tuple(block_rows)

        if block_key not in block_dates:
            continue

        block_date = block_dates[block_key]
        processed_blocks.append(block_rows)

        if PRINT_DEBUG_BLOCKS:
            print(f"Block {block_rows}, Datum {block_date}")

        for col in range(datum_col, end_col + 1):
            unique_values = get_unique_values_in_block(
                raw_snapshot=raw_snapshot,
                expanded_snapshot=expanded_snapshot,
                block_rows=block_rows,
                col=col,
                datum_col=datum_col,
                block_date=block_date
            )

            if len(unique_values) == 1:
                fill_value = unique_values[0]

                for row in block_rows:
                    cell = ws.cell(row, col)

                    if isinstance(cell, MergedCell):
                        continue

                    if col == datum_col:
                        # Niemals ein Originaldatum ersetzen.
                        can_write = (
                            is_empty(snapshot_value(raw_snapshot, row, datum_col))
                            and is_empty(cell.value)
                        )
                    else:
                        can_write = is_empty(cell.value)

                    if can_write:
                        cell.value = fill_value

                        if col == datum_col:
                            cell.number_format = "dd.mm.yyyy"

            elif len(unique_values) > 1:
                conflicts.append(
                    {
                        "rows": f"{block_rows[0]}-{block_rows[-1]}",
                        "column": ws.cell(header_row, col).value,
                        "values": unique_values,
                    }
                )

    return blocks, processed_blocks, conflicts


def main():
    wb = load_workbook(INPUT_FILE)

    if SHEET_NAME:
        ws = wb[SHEET_NAME]
    else:
        ws = wb.active

    blocks, processed_blocks, conflicts = fill_blocks(ws)

    wb.save(OUTPUT_FILE)

    print(f"Fertig. Datei gespeichert als: {OUTPUT_FILE}")
    print(f"Erkannte 3er-Blöcke: {len(blocks)}")
    print(f"Gefüllte 3er-Blöcke: {len(processed_blocks)}")

    if conflicts:
        print("\nHinweis: Einige Blöcke/Spalten wurden nicht gefüllt:")
        for conflict in conflicts[:80]:
            print(
                f"Zeilen {conflict['rows']}, "
                f"Spalte '{conflict['column']}': "
                f"{conflict['values']}"
            )


if __name__ == "__main__":
    main()

C:\Users\Felix Husmann\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


Fertig. Datei gespeichert als: ../data/raw/mta2024to2026/raw_unmerged/Störliste STW-Mittelteilanlage 2026_filled.xlsx
Erkannte 3er-Blöcke: 1709
Gefüllte 3er-Blöcke: 1709

Hinweis: Einige Blöcke/Spalten wurden nicht gefüllt:
Zeilen 3-5, Spalte 'Wochentag': ['=WEEKDAY(C3,2)', '=WEEKDAY(C4,2)', '=WEEKDAY(C5,2)']
Zeilen 3-5, Spalte 'DatumNEU': ['=IF(A23="",C2,A23)', '=IF(A24="",C3,A24)', '=IF(A25="",C4,A25)']
Zeilen 3-5, Spalte 'KW': ['=YEAR(C3)&"/"&TEXT(WEEKNUM(C3,21),"00")', '=YEAR(C4)&"/"&TEXT(WEEKNUM(C4,21),"00")', '=YEAR(C5)&"/"&TEXT(WEEKNUM(C5,21),"00")']
Zeilen 6-8, Spalte 'Wochentag': ['=WEEKDAY(C6,2)', '=WEEKDAY(C7,2)', '=WEEKDAY(C8,2)']
Zeilen 6-8, Spalte 'DatumNEU': ['=IF(A26="",C5,A26)', '=IF(#REF!="",C6,#REF!)', '=IF(#REF!="",C7,#REF!)']
Zeilen 6-8, Spalte 'KW': ['=YEAR(C6)&"/"&TEXT(WEEKNUM(C6,21),"00")', '=YEAR(C7)&"/"&TEXT(WEEKNUM(C7,21),"00")', '=YEAR(C8)&"/"&TEXT(WEEKNUM(C8,21),"00")']
Zeilen 9-11, Spalte 'Wochentag': ['=WEEKDAY(C9,2)', '=WEEKDAY(C10,2)', '=WEEKDAY(C11,2)'